# ARA Opening Signature Research Notebook — v2

Notebook ini adalah **research/catatan eksplorasi**, bukan workflow production.

Tujuan utamanya:

1. Menggunakan `all_scores` harian sebagai universe hasil filtering model: price range 100–1000, liquidity, broksum, dll.
2. Meng-align snapshot tanggal `T` dengan snapshot tanggal bursa berikutnya `T+1` berdasarkan `date` column, bukan nama file.
3. Membuat label opening gap / near-ARA / full-ARA.
4. Mendeteksi gap dari notebook v1:
   - row `next_open = 0` harus dibuang dari label valid,
   - event sangat sedikit sehingga analisis harus dibaca sebagai case-study/manual edge exploration,
   - `score_ara` menangkap sebagian event, tetapi missed event seperti INTD perlu behavioral signature.
5. Membuat score tambahan:
   - `ara_signature_score_v1`,
   - `final_ara_watch_score_50_50`,
   - `final_ara_watch_score_60_40`,
   - `final_ara_watch_score_70_30`.

Interpretasi penting:

> Analisis ini menjawab: “Dari saham yang sudah masuk daily all_scores universe, feature dan score apa yang punya edge terhadap next-day opening gap / near-ARA?”

Bukan menjawab: “Apa penyebab ARA di seluruh saham BEI?”

In [19]:
from pathlib import Path
import glob
import json
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 100)

## 1. Configuration

Atur `INPUT_GLOBS` sesuai lokasi file Anda.

Notebook ini mendukung beberapa struktur:

```text
signal_18_may_2026_all_scores.csv
signals/daily/signal_22_may_2026/continual_model/all_scores.csv
signals/daily/signal_22_may_2026/base_model/all_scores.csv
```

In [20]:
# === EDIT THESE IF NEEDED ===
ROOT = Path(".").resolve()

INPUT_GLOBS = [
    "signal_*_all_scores.csv",
    "signals/daily/signal_*/all_scores.csv",
    "signals/daily/signal_*/*/all_scores.csv",
]

OUTPUT_DIR = Path("ara_opening_signature_research_v2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRICE_MIN = 100
PRICE_MAX = 1000

# Label thresholds based on ara_ratio = next_open_ret / ara_limit_pct
STRONG_GAP_RATIO = 0.35
NEAR_ARA_RATIO = 0.70
FULL_ARA_RATIO = 0.95

# Daily top-k choices for score evaluation
TOP_KS = [1, 3, 5, 7, 10, 15, 20]

# Minimum support for threshold/interaction discovery
MIN_SUPPORT = 20
MIN_SUPPORT_PCT = 0.03

RUN_MUTUAL_INFORMATION = True  # set False if sklearn is unavailable or too slow

## 2. Load all_scores snapshots

Kita tidak akan mengandalkan tanggal dari nama file. Tanggal yang dipakai adalah `date` column di file.

In [21]:
def collect_files(patterns):
    files = []
    for pat in patterns:
        files.extend(glob.glob(str(ROOT / pat), recursive=True))
    files = sorted(set(files))
    return [Path(f) for f in files]

files = collect_files(INPUT_GLOBS)
print(f"Found {len(files)} all_scores files")
for f in files:
    print("-", f)

if not files:
    raise FileNotFoundError("No all_scores files found. Edit INPUT_GLOBS above.")

Found 14 all_scores files
- /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/signals/daily/signal_18_may_2026/base_model/all_scores.csv
- /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/signals/daily/signal_18_may_2026/continual_model/all_scores.csv
- /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/signals/daily/signal_19_may_2026/base_model/all_scores.csv
- /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/signals/daily/signal_19_may_2026/continual_model/all_scores.csv
- /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/signals/daily/signal_20_may_2026/base_model/all_scores.csv
- /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/signals/daily/signal_20_may_2026/continual_model/all_scores.csv
- /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/signals/daily/signal_21_may_2026/base_model/all_scores.csv
- /User

In [22]:
def read_all_scores_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["source_file"] = path.name
    df["source_path"] = str(path)
    if "date" not in df.columns or "ticker" not in df.columns:
        raise ValueError(f"Missing required date/ticker columns in {path}")
    df["date"] = pd.to_datetime(df["date"]).dt.date
    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
    return df

raw = pd.concat([read_all_scores_file(f) for f in files], ignore_index=True, sort=False)
print("raw shape:", raw.shape)
print("date range:", raw["date"].min(), "→", raw["date"].max())
print("unique dates:", raw["date"].nunique())
print("unique tickers:", raw["ticker"].nunique())

# If there are duplicated ticker-date rows from multiple folders/profiles, keep last by source_path order.
raw = raw.sort_values(["date", "ticker", "source_path"]).drop_duplicates(["date", "ticker"], keep="last").reset_index(drop=True)
print("after ticker-date dedup:", raw.shape)
print(raw[["date", "ticker", "source_file"]].head())

raw shape: (3152, 214)
date range: 2026-05-13 → 2026-05-25
unique dates: 7
unique tickers: 327
after ticker-date dedup: (1576, 214)
         date ticker     source_file
0  2026-05-13   ACES  all_scores.csv
1  2026-05-13   ADHI  all_scores.csv
2  2026-05-13   AGRO  all_scores.csv
3  2026-05-13   AHAP  all_scores.csv
4  2026-05-13   ALII  all_scores.csv


## 3. Build next-day opening label

Untuk setiap `ticker` pada tanggal `T`, kita ambil row ticker yang sama pada **tanggal bursa berikutnya yang tersedia dalam all_scores snapshot**.

Kelemahan mode ini:

- Jika ticker tidak muncul di snapshot berikutnya karena tidak lolos filter universe, label menjadi missing.
- Untuk production research final, `next_open` sebaiknya diambil dari canonical OHLCV, bukan dari `all_scores` berikutnya.

Namun untuk catatan riset awal, pendekatan ini sesuai dengan tujuan Anda: mengevaluasi edge di universe all_scores yang sudah tersaring.

In [23]:
REQUIRED_PRICE_COLS = ["open", "high", "low", "close"]
missing = [c for c in REQUIRED_PRICE_COLS if c not in raw.columns]
if missing:
    raise ValueError(f"Missing OHLC columns: {missing}")

# next values per ticker based on available snapshot date sequence
raw_sorted = raw.sort_values(["ticker", "date"]).copy()
for c in ["date", "open", "high", "low", "close", "volume", "value", "traded_value_proxy"]:
    if c in raw_sorted.columns:
        raw_sorted[f"next_{c}"] = raw_sorted.groupby("ticker")[c].shift(-1)

# Make next_date date-like
raw_sorted["next_date"] = pd.to_datetime(raw_sorted["next_date"]).dt.date

# Basic label fields
df = raw_sorted.copy()
for c in ["open", "high", "low", "close", "next_open", "next_high", "next_low", "next_close"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Price universe filter on T close, consistent with your daily all_scores objective.
df = df[(df["close"] >= PRICE_MIN) & (df["close"] <= PRICE_MAX)].copy()

print("rows after price filter:", df.shape)
print("next_open null:", df["next_open"].isna().sum())
print("next_open <= 0:", (df["next_open"].fillna(0) <= 0).sum())

rows after price filter: (1576, 222)
next_open null: 327
next_open <= 0: 411


## 4. ARA limit and tick-adjusted ARA price

Aturan ARA yang dipakai:

```text
Rp50–Rp200      : +35%
>Rp200–Rp5,000  : +25%
>Rp5,000        : +20%
```

Karena harga BEI menggunakan tick size, kita tambahkan estimasi tick-adjusted ARA price. Untuk research awal, label utama tetap menggunakan `ara_ratio`, tetapi tick-adjusted fields berguna untuk audit case-by-case.

In [24]:
def idx_tick_size(price):
    if pd.isna(price):
        return np.nan
    p = float(price)
    if p < 200:
        return 1
    if p < 500:
        return 2
    if p < 2000:
        return 5
    if p < 5000:
        return 10
    return 25


def floor_to_tick(price, tick):
    if pd.isna(price) or pd.isna(tick) or tick <= 0:
        return np.nan
    return math.floor(float(price) / float(tick)) * float(tick)


def ara_limit_pct(close):
    if pd.isna(close):
        return np.nan
    c = float(close)
    if 50 <= c <= 200:
        return 0.35
    if 200 < c <= 5000:
        return 0.25
    if c > 5000:
        return 0.20
    return np.nan


df["ara_limit_pct"] = df["close"].apply(ara_limit_pct)
df["tick_size"] = df["close"].apply(idx_tick_size)
df["raw_ara_price"] = df["close"] * (1 + df["ara_limit_pct"])
df["ara_price_tick_adjusted"] = [floor_to_tick(p, t) for p, t in zip(df["raw_ara_price"], df["tick_size"])]
df["tick_distance_to_ara_open"] = (df["ara_price_tick_adjusted"] - df["next_open"]) / df["tick_size"]

df["next_open_ret"] = (df["next_open"] / df["close"]) - 1
df["ara_ratio"] = df["next_open_ret"] / df["ara_limit_pct"]

# Strict valid label filter: next_open must be positive.
df["valid_opening_label"] = (
    df["next_open"].notna() &
    (df["next_open"] > 0) &
    df["close"].notna() &
    (df["close"] > 0) &
    df["ara_limit_pct"].notna()
)

# Class labels. Invalid label rows are set to missing_label.
def classify_ara_ratio(r):
    if pd.isna(r):
        return "missing_label"
    if r >= FULL_ARA_RATIO:
        return "full_ara_opening"
    if r >= NEAR_ARA_RATIO:
        return "near_ara"
    if r >= STRONG_GAP_RATIO:
        return "strong_gap_up"
    return "normal_movement"

# tick-adjusted full ARA alternative
# full_ara_tick_adjusted = next_open >= ara adjusted price - 1 tick tolerance
df["is_full_ara_tick_adjusted"] = (
    df["valid_opening_label"] &
    (df["next_open"] >= (df["ara_price_tick_adjusted"] - df["tick_size"]))
).astype(int)

df.loc[df["valid_opening_label"], "ara_opening_class"] = df.loc[df["valid_opening_label"], "ara_ratio"].apply(classify_ara_ratio)
df.loc[~df["valid_opening_label"], "ara_opening_class"] = "missing_label"

df["is_strong_gap_or_better"] = df["ara_opening_class"].isin(["strong_gap_up", "near_ara", "full_ara_opening"]).astype(int)
df["is_near_or_full_ara"] = df["ara_opening_class"].isin(["near_ara", "full_ara_opening"]).astype(int)
df["is_full_ara_opening"] = df["ara_opening_class"].eq("full_ara_opening").astype(int)

valid = df[df["valid_opening_label"]].copy()
print("valid labeled rows:", valid.shape)
print(valid["ara_opening_class"].value_counts(dropna=False))
print("invalid/missing label rows:", (~df["valid_opening_label"]).sum())

valid labeled rows: (1165, 235)
ara_opening_class
normal_movement    1160
strong_gap_up         4
near_ara              1
Name: count, dtype: int64
invalid/missing label rows: 411


## 5. Data-quality audit

Bagian ini khusus untuk menghindari bias notebook v1: `next_open = 0` dapat menciptakan return -100% palsu dan merusak threshold/statistics.

In [25]:
dq = {
    "raw_rows_after_price_filter": int(len(df)),
    "valid_labeled_rows": int(len(valid)),
    "missing_next_open": int(df["next_open"].isna().sum()),
    "next_open_le_zero": int((df["next_open"].fillna(0) <= 0).sum()),
    "valid_date_min": str(valid["date"].min()) if len(valid) else None,
    "valid_date_max": str(valid["date"].max()) if len(valid) else None,
    "valid_unique_dates": int(valid["date"].nunique()),
    "valid_unique_tickers": int(valid["ticker"].nunique()),
}
print(json.dumps(dq, indent=2))

pd.Series(dq).to_frame("value").to_csv(OUTPUT_DIR / "data_quality_audit.csv")

{
  "raw_rows_after_price_filter": 1576,
  "valid_labeled_rows": 1165,
  "missing_next_open": 327,
  "next_open_le_zero": 411,
  "valid_date_min": "2026-05-13",
  "valid_date_max": "2026-05-22",
  "valid_unique_dates": 6,
  "valid_unique_tickers": 259
}


## 6. Event list: strong gap or better

Ini adalah event utama yang kita gunakan untuk case-study manual.

In [26]:
event_cols = [
    "date", "ticker", "next_date", "close", "next_open", "next_open_ret", "ara_limit_pct", "ara_ratio",
    "tick_size", "ara_price_tick_adjusted", "tick_distance_to_ara_open", "ara_opening_class",
]
for c in ["score_ara", "score_scalp", "score_swing", "score_momentum_5d", "score_momentum_10d", "score_position"]:
    if c in valid.columns and c not in event_cols:
        event_cols.append(c)

strong_events = valid[valid["is_strong_gap_or_better"] == 1].sort_values("ara_ratio", ascending=False)
strong_events[event_cols].to_csv(OUTPUT_DIR / "strong_gap_or_better_events_v2.csv", index=False)
display(strong_events[event_cols])

,date,ticker,next_date,close,next_open,next_open_ret,ara_limit_pct,ara_ratio,tick_size,ara_price_tick_adjusted,tick_distance_to_ara_open,ara_opening_class,score_ara,score_scalp,score_swing,score_momentum_5d,score_momentum_10d,score_position
833,2026-05-20,INTD,2026-05-21,286.0,344.0,0.202797,0.25,0.811189,2,356.0,6.0,near_ara,0.050528,0.078807,0.131728,0.131728,0.243809,0.183312
77,2026-05-13,FITT,2026-05-21,410.0,466.0,0.136585,0.25,0.546341,2,512.0,23.0,strong_gap_up,0.094538,0.213595,0.170297,0.170297,0.273676,0.190734
1239,2026-05-22,DIVA,2026-05-25,136.0,160.0,0.176471,0.35,0.504202,1,183.0,23.0,strong_gap_up,0.876316,0.149909,0.142572,0.142572,0.173916,0.209124
1370,2026-05-22,TALF,2026-05-25,780.0,860.0,0.102564,0.25,0.410256,5,975.0,23.0,strong_gap_up,0.051335,0.107750,0.148780,0.148780,0.268312,0.224923
362,2026-05-18,LCKM,2026-05-19,112.0,127.0,0.133929,0.35,0.382653,1,151.0,24.0,strong_gap_up,0.090767,0.155936,0.150023,0.150023,0.260669,0.189953


## 7. Feature candidates and rank normalization

Kita buat `ara_signature_score_v1` dari behavioral features yang tersedia. Semua feature dirank **cross-sectionally per date** agar skala antar-feature lebih comparable.

Intuition score:

```text
volume expansion
+ recent momentum
+ close above MA20
+ volatility opening up
+ buyer dominance
+ net flow
+ rank1 buyer persistence
```

In [27]:
def available(cols):
    return [c for c in cols if c in valid.columns]

behavior_features = available([
    "volume_ratio_20d",
    "ret_1d",
    "ret_5d",
    "ret_10d",
    "close_vs_ma20",
    "volatility_20d",
    "buyer_dominance_ratio",
    "net_flow_ratio",
    "rank1_same_buyer_streak",
    "buy_val_total",
    "net_val_total",
    "traded_value_proxy",
])

score_cols = [c for c in valid.columns if c.startswith("score_")]
print("behavior_features:", behavior_features)
print("score_cols:", score_cols[:50], "... total", len(score_cols))

work = valid.copy()

# Create per-date percentile rank columns. Higher is assumed better.
for c in behavior_features + score_cols:
    x = pd.to_numeric(work[c], errors="coerce")
    work[c] = x
    work[f"rank_{c}"] = work.groupby("date")[c].rank(pct=True, method="average")

# Weighted behavioral signature. Use only available rank columns.
weights = {
    "rank_volume_ratio_20d": 0.22,
    "rank_ret_1d": 0.18,
    "rank_ret_5d": 0.12,
    "rank_close_vs_ma20": 0.14,
    "rank_volatility_20d": 0.12,
    "rank_buyer_dominance_ratio": 0.12,
    "rank_net_flow_ratio": 0.07,
    "rank_rank1_same_buyer_streak": 0.03,
}

num = 0
score = pd.Series(0.0, index=work.index)
used_weights = {}
for col, w in weights.items():
    if col in work.columns:
        vals = work[col].fillna(0.5)
        score += w * vals
        num += w
        used_weights[col] = w

if num > 0:
    work["ara_signature_score_v1"] = score / num
else:
    work["ara_signature_score_v1"] = np.nan

# Rank signature per day.
work["rank_ara_signature_score_v1"] = work.groupby("date")["ara_signature_score_v1"].rank(pct=True, method="average")

# Model score rank. If score_ara missing, fallback to best ara-like score.
if "score_ara" in work.columns:
    ara_model_score_col = "score_ara"
else:
    ara_like = [c for c in score_cols if "ara" in c.lower()]
    ara_model_score_col = ara_like[0] if ara_like else None

print("used_weights:", used_weights)
print("ara_model_score_col:", ara_model_score_col)

if ara_model_score_col:
    if f"rank_{ara_model_score_col}" not in work.columns:
        work[f"rank_{ara_model_score_col}"] = work.groupby("date")[ara_model_score_col].rank(pct=True, method="average")
    model_rank = work[f"rank_{ara_model_score_col}"].fillna(0.5)
else:
    model_rank = pd.Series(0.5, index=work.index)

sig_rank = work["rank_ara_signature_score_v1"].fillna(0.5)
work["final_ara_watch_score_50_50"] = 0.50 * model_rank + 0.50 * sig_rank
work["final_ara_watch_score_60_40"] = 0.60 * model_rank + 0.40 * sig_rank
work["final_ara_watch_score_70_30"] = 0.70 * model_rank + 0.30 * sig_rank

# Rank final scores per day too.
for c in ["final_ara_watch_score_50_50", "final_ara_watch_score_60_40", "final_ara_watch_score_70_30"]:
    work[f"rank_{c}"] = work.groupby("date")[c].rank(pct=True, method="average")

print(work[["date", "ticker", "ara_signature_score_v1", "final_ara_watch_score_50_50"]].head())

behavior_features: ['volume_ratio_20d', 'ret_1d', 'ret_5d', 'ret_10d', 'close_vs_ma20', 'volatility_20d', 'buyer_dominance_ratio', 'net_flow_ratio', 'rank1_same_buyer_streak', 'buy_val_total', 'net_val_total', 'traded_value_proxy']
score_cols: ['score_sm', 'score_ara', 'score_mm_silent', 'score_momentum_5d__momentum_ranker__xgb__momentum_5d', 'score_momentum_10d__momentum_ranker__hgb__momentum_10d', 'score_scalp__multi_strategy_time__rank_hgb__scalp', 'score_swing__multi_strategy_time__hgb__swing', 'score_position__multi_strategy_time__xgb__position', 'score_momentum_5d', 'score_momentum_10d', 'score_momentum_20d', 'score_scalp', 'score_swing', 'score_position'] ... total 14
used_weights: {'rank_volume_ratio_20d': 0.22, 'rank_ret_1d': 0.18, 'rank_ret_5d': 0.12, 'rank_close_vs_ma20': 0.14, 'rank_volatility_20d': 0.12, 'rank_buyer_dominance_ratio': 0.12, 'rank_net_flow_ratio': 0.07, 'rank_rank1_same_buyer_streak': 0.03}
ara_model_score_col: score_ara
           date ticker  ara_signature

## 8. Daily top-k edge evaluation

Di sini kita evaluasi apakah score tertentu lebih sering menangkap `strong_gap_or_better` pada top-k harian.

Catatan: karena event sangat sedikit, fokus pada:

- `strong_gap_rate`,
- `near_or_full_ara_rate`,
- event capture table,
- bukan sekadar average return.

In [28]:
def topk_edge_evaluation(data, score_columns, topks=TOP_KS):
    rows = []
    baseline_strong = data["is_strong_gap_or_better"].mean()
    baseline_near = data["is_near_or_full_ara"].mean()
    for score in score_columns:
        if score not in data.columns:
            continue
        tmp = data[data[score].notna()].copy()
        if tmp.empty:
            continue
        tmp["rank_desc"] = tmp.groupby("date")[score].rank(ascending=False, method="first")
        for k in topks:
            sel = tmp[tmp["rank_desc"] <= k]
            if sel.empty:
                continue
            rows.append({
                "score_col": score,
                "top_k": k,
                "n_candidates": int(len(sel)),
                "n_days": int(sel["date"].nunique()),
                "avg_next_open_ret": sel["next_open_ret"].mean(),
                "median_next_open_ret": sel["next_open_ret"].median(),
                "avg_ara_ratio": sel["ara_ratio"].mean(),
                "p90_ara_ratio": sel["ara_ratio"].quantile(0.90),
                "strong_gap_rate": sel["is_strong_gap_or_better"].mean(),
                "near_or_full_ara_rate": sel["is_near_or_full_ara"].mean(),
                "event_capture_strong": int(sel["is_strong_gap_or_better"].sum()),
                "event_capture_near": int(sel["is_near_or_full_ara"].sum()),
                "lift_strong_gap_vs_baseline": sel["is_strong_gap_or_better"].mean() / baseline_strong if baseline_strong > 0 else np.nan,
                "lift_near_ara_vs_baseline": sel["is_near_or_full_ara"].mean() / baseline_near if baseline_near > 0 else np.nan,
            })
    return pd.DataFrame(rows).sort_values(["event_capture_strong", "strong_gap_rate", "top_k"], ascending=[False, False, True])

candidate_scores = []
for c in [
    "score_ara",
    "ara_signature_score_v1",
    "final_ara_watch_score_50_50",
    "final_ara_watch_score_60_40",
    "final_ara_watch_score_70_30",
]:
    if c in work.columns:
        candidate_scores.append(c)

# Add all score_* columns for comparison, but keep output sorted.
candidate_scores += [c for c in score_cols if c not in candidate_scores]

edge = topk_edge_evaluation(work, candidate_scores)
edge.to_csv(OUTPUT_DIR / "score_topk_edge_evaluation_v2.csv", index=False)
display(edge.head(50))

,score_col,top_k,n_candidates,n_days,avg_next_open_ret,median_next_open_ret,avg_ara_ratio,p90_ara_ratio,strong_gap_rate,near_or_full_ara_rate,event_capture_strong,event_capture_near,lift_strong_gap_vs_baseline,lift_near_ara_vs_baseline
12,ara_signature_score_v1,15,90,6,0.007622,0.000000,0.027591,0.109789,0.033333,0.011111,3,1,7.766667,12.944444
13,ara_signature_score_v1,20,120,6,0.002540,0.000000,0.010240,0.109789,0.025000,0.008333,3,1,5.825000,9.708333
8,ara_signature_score_v1,3,18,6,0.020417,0.000000,0.080044,0.189219,0.111111,0.055556,2,1,25.888889,64.722222
9,ara_signature_score_v1,5,30,6,0.015598,0.002577,0.061642,0.116025,0.066667,0.033333,2,1,15.533333,38.833333
10,ara_signature_score_v1,7,42,6,0.017326,0.005702,0.063314,0.139500,0.047619,0.023810,2,1,11.095238,27.738095
11,ara_signature_score_v1,10,60,6,0.012272,0.004167,0.044707,0.105666,0.033333,0.016667,2,1,7.766667,19.416667
0,score_ara,1,6,6,0.003783,-0.003546,-0.013361,0.305434,0.166667,0.000000,1,0,38.833333,0.000000
1,score_ara,3,18,6,-0.014094,0.000000,-0.063663,0.130479,0.055556,0.000000,1,0,12.944444,0.000000
2,score_ara,5,30,6,-0.014673,0.000000,-0.057172,0.118453,0.033333,0.000000,1,0,7.766667,0.000000
30,final_ara_watch_score_70_30,5,30,6,-0.022706,0.000000,-0.081271,0.088041,0.033333,0.000000,1,0,7.766667,0.000000


## 9. Event capture / missed-event analysis

Tujuan bagian ini: menjawab pertanyaan “event seperti INTD missed oleh `score_ara`, tetapi apakah tertangkap oleh `ara_signature_score` atau combined score?”

In [29]:
def rank_events_by_scores(data, events, scores):
    rows = []
    for _, ev in events.iterrows():
        day = data[data["date"] == ev["date"]].copy()
        for s in scores:
            if s not in day.columns or day[s].isna().all():
                continue
            day[f"rank_{s}_desc"] = day[s].rank(ascending=False, method="first")
            hit = day[day["ticker"] == ev["ticker"]]
            if hit.empty:
                continue
            rank = int(hit[f"rank_{s}_desc"].iloc[0])
            value = hit[s].iloc[0]
            rows.append({
                "date": ev["date"],
                "ticker": ev["ticker"],
                "ara_opening_class": ev["ara_opening_class"],
                "next_open_ret": ev["next_open_ret"],
                "ara_ratio": ev["ara_ratio"],
                "score_col": s,
                "score_value": value,
                "daily_rank_desc": rank,
                "daily_percentile_desc": 1 - ((rank - 1) / max(len(day), 1)),
                "in_top_1": rank <= 1,
                "in_top_3": rank <= 3,
                "in_top_5": rank <= 5,
                "in_top_10": rank <= 10,
            })
    return pd.DataFrame(rows).sort_values(["date", "ticker", "daily_rank_desc"])

capture_scores = [s for s in [
    "score_ara",
    "ara_signature_score_v1",
    "final_ara_watch_score_50_50",
    "final_ara_watch_score_60_40",
    "final_ara_watch_score_70_30",
    "score_scalp",
    "score_swing",
    "score_momentum_5d",
] if s in work.columns]

capture = rank_events_by_scores(work, strong_events, capture_scores)
capture.to_csv(OUTPUT_DIR / "event_capture_rank_by_score_v2.csv", index=False)
display(capture)

,date,ticker,ara_opening_class,next_open_ret,ara_ratio,score_col,score_value,daily_rank_desc,daily_percentile_desc,in_top_1,in_top_3,in_top_5,in_top_10
10,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,final_ara_watch_score_50_50,0.550481,71,0.663462,False,False,False,False
13,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,score_scalp,0.213595,75,0.644231,False,False,False,False
9,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,ara_signature_score_v1,0.520024,86,0.591346,False,False,False,False
14,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,score_swing,0.170297,86,0.591346,False,False,False,False
15,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,score_momentum_5d,0.170297,86,0.591346,False,False,False,False
11,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,final_ara_watch_score_60_40,0.542308,90,0.572115,False,False,False,False
12,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,final_ara_watch_score_70_30,0.534135,99,0.528846,False,False,False,False
8,2026-05-13,FITT,strong_gap_up,0.136585,0.546341,score_ara,0.094538,103,0.509615,False,False,False,False
33,2026-05-18,LCKM,strong_gap_up,0.133929,0.382653,ara_signature_score_v1,0.755752,13,0.941748,False,False,False,False
34,2026-05-18,LCKM,strong_gap_up,0.133929,0.382653,final_ara_watch_score_50_50,0.720874,24,0.888350,False,False,False,False


## 10. Feature distribution: event vs non-event

Kita bandingkan median event strong/near dengan median non-event.

In [30]:
analysis_features = available([
    "score_ara",
    "ara_signature_score_v1",
    "final_ara_watch_score_50_50",
    "ret_1d", "ret_5d", "ret_10d", "ret_20d",
    "volume_ratio_20d", "volume", "value", "traded_value_proxy",
    "close_vs_ma20", "volatility_20d",
    "buyer_dominance_ratio", "net_flow_ratio", "rank1_same_buyer_streak",
    "net_val_total", "buy_val_total", "sell_val_total",
])

rows=[]
for feat in analysis_features:
    x_event = pd.to_numeric(work.loc[work["is_strong_gap_or_better"]==1, feat], errors="coerce")
    x_non = pd.to_numeric(work.loc[work["is_strong_gap_or_better"]==0, feat], errors="coerce")
    rows.append({
        "feature": feat,
        "event_median": x_event.median(),
        "non_event_median": x_non.median(),
        "event_mean": x_event.mean(),
        "non_event_mean": x_non.mean(),
        "event_non_null": int(x_event.notna().sum()),
        "non_event_non_null": int(x_non.notna().sum()),
        "median_diff": x_event.median() - x_non.median(),
        "median_ratio_event_vs_non": x_event.median() / x_non.median() if pd.notna(x_non.median()) and x_non.median()!=0 else np.nan,
    })

dist = pd.DataFrame(rows).sort_values("median_diff", ascending=False)
dist.to_csv(OUTPUT_DIR / "feature_distribution_event_vs_non_event_v2.csv", index=False)
display(dist)

,feature,event_median,non_event_median,event_mean,non_event_mean,event_non_null,non_event_non_null,median_diff,median_ratio_event_vs_non
5,volume_ratio_20d,2.068619e+00,8.440464e-01,2.109630e+00,1.263091e+00,5,1160,1.224573e+00,2.450836
13,rank1_same_buyer_streak,2.000000e+00,1.000000e+00,5.400000e+00,5.190517e+00,5,1160,1.000000e+00,2.000000
4,ret_20d,2.380952e-01,-8.000000e-02,2.017894e-01,-4.184660e-02,5,1160,3.180952e-01,-2.976190
11,buyer_dominance_ratio,4.390495e-01,2.806706e-01,4.345474e-01,3.158084e-01,5,1160,1.583789e-01,1.564287
2,ret_5d,4.901961e-03,-9.090909e-02,-5.998973e-03,-8.933450e-02,5,1160,9.581105e-02,-0.053922
10,volatility_20d,1.077751e-01,4.196772e-02,9.484059e-02,4.840454e-02,5,1160,6.580742e-02,2.568048
9,close_vs_ma20,-4.109589e-02,-9.428079e-02,9.645217e-06,-9.245706e-02,5,1160,5.318490e-02,0.435888
1,ret_1d,2.752294e-02,-2.339181e-02,7.087290e-02,-2.620685e-02,5,1160,5.091475e-02,-1.176606
12,net_flow_ratio,0.000000e+00,-2.530819e-05,-2.002199e-04,-6.561732e-04,5,1160,2.530819e-05,-0.000000
0,score_ara,9.076696e-02,9.849148e-02,2.326969e-01,2.224164e-01,5,1160,-7.724522e-03,0.921572


## 11. Probabilistic threshold discovery

Cari threshold feature yang meningkatkan probability `strong_gap_or_better`.

Karena sample pendek, threshold ini bukan final rule, tetapi kandidat riset.

In [31]:
def threshold_discovery(data, features, quantiles=(0.50,0.60,0.70,0.75,0.80,0.85,0.90,0.95), target="is_strong_gap_or_better"):
    rows=[]
    baseline = data[target].mean()
    baseline_near = data["is_near_or_full_ara"].mean()
    n_total = len(data)
    for feat in features:
        x = pd.to_numeric(data[feat], errors="coerce")
        if x.notna().sum() < MIN_SUPPORT:
            continue
        for q in quantiles:
            th = x.quantile(q)
            if pd.isna(th):
                continue
            for direction in [">=", "<="]:
                if direction == ">=":
                    mask = x >= th
                else:
                    mask = x <= th
                support = int(mask.sum())
                if support < MIN_SUPPORT or support / n_total < MIN_SUPPORT_PCT:
                    continue
                sel = data[mask]
                rows.append({
                    "feature": feat,
                    "direction": direction,
                    "quantile": q,
                    "threshold": th,
                    "support": support,
                    "support_pct": support / n_total,
                    "avg_next_open_ret": sel["next_open_ret"].mean(),
                    "median_next_open_ret": sel["next_open_ret"].median(),
                    "avg_ara_ratio": sel["ara_ratio"].mean(),
                    "strong_gap_rate": sel["is_strong_gap_or_better"].mean(),
                    "near_or_full_ara_rate": sel["is_near_or_full_ara"].mean(),
                    "event_capture_strong": int(sel["is_strong_gap_or_better"].sum()),
                    "event_capture_near": int(sel["is_near_or_full_ara"].sum()),
                    "lift_strong_gap": sel["is_strong_gap_or_better"].mean() / baseline if baseline > 0 else np.nan,
                    "lift_near_or_full_ara": sel["is_near_or_full_ara"].mean() / baseline_near if baseline_near > 0 else np.nan,
                })
    out = pd.DataFrame(rows)
    if len(out):
        out = out.sort_values(["event_capture_strong", "strong_gap_rate", "lift_strong_gap", "support"], ascending=[False, False, False, False])
    return out

threshold_features = available([
    "ara_signature_score_v1", "final_ara_watch_score_50_50", "final_ara_watch_score_60_40", "final_ara_watch_score_70_30",
    "score_ara", "score_scalp", "score_swing", "score_momentum_5d",
    "volume_ratio_20d", "ret_1d", "ret_5d", "ret_10d", "close_vs_ma20", "volatility_20d",
    "buyer_dominance_ratio", "net_flow_ratio", "rank1_same_buyer_streak", "traded_value_proxy"
])

thr = threshold_discovery(work, threshold_features)
thr.to_csv(OUTPUT_DIR / "probabilistic_threshold_discovery_v2.csv", index=False)
display(thr.head(50))

,feature,direction,quantile,threshold,support,support_pct,avg_next_open_ret,median_next_open_ret,avg_ara_ratio,strong_gap_rate,near_or_full_ara_rate,event_capture_strong,event_capture_near,lift_strong_gap,lift_near_or_full_ara
148,volatility_20d,>=,0.70,5.634666e-02,350,0.300429,-0.000389,0.0,-0.002293,0.014286,0.002857,5,1,3.328571,3.328571
146,volatility_20d,>=,0.60,4.888824e-02,466,0.400000,0.000138,0.0,0.000056,0.010730,0.002146,5,1,2.500000,2.500000
162,buyer_dominance_ratio,>=,0.60,3.191237e-01,466,0.400000,0.001771,0.0,0.007328,0.010730,0.002146,5,1,2.500000,2.500000
144,volatility_20d,>=,0.50,4.209183e-02,583,0.500429,0.000128,0.0,0.000127,0.008576,0.001715,5,1,1.998285,1.998285
160,buyer_dominance_ratio,>=,0.50,2.817766e-01,583,0.500429,0.000717,0.0,0.003124,0.008576,0.001715,5,1,1.998285,1.998285
35,score_swing,<=,0.60,1.901287e-01,699,0.600000,0.001454,0.0,0.005520,0.007153,0.001431,5,1,1.666667,1.666667
51,score_momentum_5d,<=,0.60,1.901287e-01,699,0.600000,0.001454,0.0,0.005520,0.007153,0.001431,5,1,1.666667,1.666667
37,score_swing,<=,0.70,2.306256e-01,815,0.699571,0.000748,0.0,0.003034,0.006135,0.001227,5,1,1.429448,1.429448
53,score_momentum_5d,<=,0.70,2.306256e-01,815,0.699571,0.000748,0.0,0.003034,0.006135,0.001227,5,1,1.429448,1.429448
23,score_scalp,<=,0.75,2.922382e-01,874,0.750215,0.002483,0.0,0.008978,0.005721,0.001144,5,1,1.332952,1.332952


## 12. Simple interaction discovery

Kita cari kombinasi dua feature yang punya lift lebih tinggi daripada masing-masing feature sendiri.

In [32]:
def interaction_discovery(data, features, q=0.75, target="is_strong_gap_or_better"):
    rows=[]
    baseline = data[target].mean()
    baseline_near = data["is_near_or_full_ara"].mean()
    n_total = len(data)
    vals = {f: pd.to_numeric(data[f], errors="coerce") for f in features}
    ths = {f: vals[f].quantile(q) for f in features if vals[f].notna().sum() >= MIN_SUPPORT}
    fs = list(ths.keys())
    for i, a in enumerate(fs):
        for b in fs[i+1:]:
            mask = (vals[a] >= ths[a]) & (vals[b] >= ths[b])
            support = int(mask.sum())
            if support < MIN_SUPPORT or support / n_total < MIN_SUPPORT_PCT:
                continue
            sel = data[mask]
            rows.append({
                "feature_a": a,
                "feature_b": b,
                "condition": f"{a} >= p{int(q*100)} AND {b} >= p{int(q*100)}",
                "threshold_a": ths[a],
                "threshold_b": ths[b],
                "support": support,
                "support_pct": support / n_total,
                "avg_next_open_ret": sel["next_open_ret"].mean(),
                "median_next_open_ret": sel["next_open_ret"].median(),
                "avg_ara_ratio": sel["ara_ratio"].mean(),
                "strong_gap_rate": sel["is_strong_gap_or_better"].mean(),
                "near_or_full_ara_rate": sel["is_near_or_full_ara"].mean(),
                "event_capture_strong": int(sel["is_strong_gap_or_better"].sum()),
                "event_capture_near": int(sel["is_near_or_full_ara"].sum()),
                "lift_strong_gap": sel["is_strong_gap_or_better"].mean() / baseline if baseline > 0 else np.nan,
                "lift_near_or_full_ara": sel["is_near_or_full_ara"].mean() / baseline_near if baseline_near > 0 else np.nan,
            })
    out = pd.DataFrame(rows)
    if len(out):
        out = out.sort_values(["event_capture_strong", "strong_gap_rate", "lift_strong_gap", "support"], ascending=[False, False, False, False])
    return out

interaction_features = available([
    "ara_signature_score_v1", "score_ara",
    "volume_ratio_20d", "ret_1d", "ret_5d", "close_vs_ma20", "volatility_20d",
    "buyer_dominance_ratio", "net_flow_ratio", "rank1_same_buyer_streak",
])

inter = interaction_discovery(work, interaction_features, q=0.75)
inter.to_csv(OUTPUT_DIR / "simple_feature_interaction_discovery_v2.csv", index=False)
display(inter.head(50))

,feature_a,feature_b,condition,threshold_a,threshold_b,support,support_pct,avg_next_open_ret,median_next_open_ret,avg_ara_ratio,strong_gap_rate,near_or_full_ara_rate,event_capture_strong,event_capture_near,lift_strong_gap,lift_near_or_full_ara
8,volume_ratio_20d,volatility_20d,volume_ratio_20d >= p75 AND volatility_20d >= p75,1.442662,0.061885,70,0.060086,-0.005043,0.000000,-0.019839,0.042857,0.014286,3,1,9.985714,16.642857
27,volatility_20d,buyer_dominance_ratio,volatility_20d >= p75 AND buyer_dominance_rati...,0.061885,0.387923,73,0.062661,0.003435,0.000000,0.015473,0.041096,0.013699,3,1,9.575342,15.958904
19,ret_5d,volatility_20d,ret_5d >= p75 AND volatility_20d >= p75,-0.023077,0.061885,84,0.072103,0.001992,0.000000,0.011557,0.035714,0.011905,3,1,8.321429,13.869048
14,ret_1d,volatility_20d,ret_1d >= p75 AND volatility_20d >= p75,0.000000,0.061885,97,0.083262,0.014366,0.006667,0.048416,0.030928,0.010309,3,1,7.206186,12.010309
9,volume_ratio_20d,buyer_dominance_ratio,volume_ratio_20d >= p75 AND buyer_dominance_ra...,1.442662,0.387923,102,0.087554,0.003673,0.000000,0.014320,0.029412,0.009804,3,1,6.852941,11.421569
15,ret_1d,buyer_dominance_ratio,ret_1d >= p75 AND buyer_dominance_ratio >= p75,0.000000,0.387923,108,0.092704,0.007521,0.000000,0.028108,0.027778,0.009259,3,1,6.472222,10.787037
5,volume_ratio_20d,ret_1d,volume_ratio_20d >= p75 AND ret_1d >= p75,1.442662,0.000000,118,0.101288,0.006405,0.000000,0.023369,0.025424,0.008475,3,1,5.923729,9.872881
30,buyer_dominance_ratio,net_flow_ratio,buyer_dominance_ratio >= p75 AND net_flow_rati...,0.387923,0.000046,70,0.060086,0.007486,0.000000,0.026134,0.028571,0.014286,2,1,6.657143,16.642857
10,volume_ratio_20d,net_flow_ratio,volume_ratio_20d >= p75 AND net_flow_ratio >= p75,1.442662,0.000046,92,0.078970,-0.000498,0.000000,-0.004552,0.021739,0.010870,2,1,5.065217,12.663043
28,volatility_20d,net_flow_ratio,volatility_20d >= p75 AND net_flow_ratio >= p75,0.061885,0.000046,95,0.081545,0.006225,0.000000,0.016199,0.021053,0.010526,2,1,4.905263,12.263158


## 13. Optional: mutual information scan

Jalankan hanya sebagai tambahan. Pada sample kecil, MI bisa noisy.

In [33]:
if RUN_MUTUAL_INFORMATION:
    try:
        from sklearn.feature_selection import mutual_info_classif
        mi_features = available([
            "ara_signature_score_v1", "final_ara_watch_score_50_50", "final_ara_watch_score_60_40", "final_ara_watch_score_70_30",
            "score_ara", "score_scalp", "score_swing", "score_momentum_5d",
            "volume_ratio_20d", "ret_1d", "ret_5d", "ret_10d", "ret_20d",
            "close_vs_ma20", "volatility_20d",
            "buyer_dominance_ratio", "net_flow_ratio", "rank1_same_buyer_streak", "traded_value_proxy"
        ])
        X = work[mi_features].apply(pd.to_numeric, errors="coerce")
        # simple imputation for MI scan
        X = X.fillna(X.median(numeric_only=True)).fillna(0)
        y = work["is_strong_gap_or_better"].astype(int)
        mi = mutual_info_classif(X, y, discrete_features=False, random_state=42)
        mi_df = pd.DataFrame({"feature": mi_features, "mutual_information": mi}).sort_values("mutual_information", ascending=False)
        mi_df.to_csv(OUTPUT_DIR / "feature_mutual_information_scan_v2.csv", index=False)
        display(mi_df)
    except Exception as e:
        print("Mutual information skipped:", repr(e))
else:
    print("RUN_MUTUAL_INFORMATION=False")

,feature,mutual_information
2,score_swing,0.006348
3,score_momentum_5d,0.006347
11,buyer_dominance_ratio,0.003415
12,net_flow_ratio,0.002907
0,score_ara,0.001799
8,ret_20d,0.001667
10,volatility_20d,0.001105
1,score_scalp,0.000749
9,close_vs_ma20,0.000604
4,volume_ratio_20d,0.000000


## 14. Spearman correlation scan

Correlation bukan alat utama untuk rare event, tetapi berguna sebagai audit cepat.

In [34]:
corr_features = available([
    "ara_signature_score_v1", "final_ara_watch_score_50_50", "final_ara_watch_score_60_40", "final_ara_watch_score_70_30",
    "score_ara", "score_scalp", "score_swing", "score_momentum_5d", "score_momentum_10d", "score_position",
    "volume_ratio_20d", "ret_1d", "ret_5d", "ret_10d", "ret_20d",
    "close_vs_ma20", "volatility_20d", "buyer_dominance_ratio", "net_flow_ratio", "rank1_same_buyer_streak", "traded_value_proxy"
])

rows=[]
for feat in corr_features:
    x = pd.to_numeric(work[feat], errors="coerce")
    rows.append({
        "feature": feat,
        "spearman_vs_next_open_ret": x.corr(work["next_open_ret"], method="spearman"),
        "spearman_vs_ara_ratio": x.corr(work["ara_ratio"], method="spearman"),
        "spearman_vs_strong_gap_target": x.corr(work["is_strong_gap_or_better"], method="spearman"),
        "non_null": int(x.notna().sum()),
    })

corr = pd.DataFrame(rows).sort_values("spearman_vs_strong_gap_target", ascending=False)
corr.to_csv(OUTPUT_DIR / "feature_spearman_correlation_scan_v2.csv", index=False)
display(corr)

,feature,spearman_vs_next_open_ret,spearman_vs_ara_ratio,spearman_vs_strong_gap_target,non_null
12,volatility_20d,0.043154,0.034790,0.083866,1165
13,buyer_dominance_ratio,0.058734,0.055842,0.071333,1165
10,ret_20d,0.047249,0.043422,0.061084,1165
11,close_vs_ma20,0.046125,0.046361,0.045447,1165
6,volume_ratio_20d,-0.038537,-0.031372,0.041269,1165
8,ret_5d,0.069240,0.069468,0.036955,1165
7,ret_1d,0.274992,0.277993,0.035456,1165
14,net_flow_ratio,0.094543,0.093302,0.025184,1165
9,ret_10d,0.039124,0.040452,0.017843,1165
5,score_position,-0.032461,-0.035668,0.013978,1165


## 15. Event-study T-minus rows

Karena sample event kecil, event-study ini lebih berguna untuk case inspection daripada aggregate statistics.

In [35]:
# Build t-minus rows from work by matching each event ticker on prior available dates.
tminus_rows = []
for _, ev in strong_events.iterrows():
    hist = (
        work[(work["ticker"] == ev["ticker"]) & (work["date"] <= ev["date"])]
        .sort_values("date", ascending=False)
        .head(6)
        .copy()
    )
    hist["event_date"] = ev["date"]
    hist["event_ticker"] = ev["ticker"]
    hist["t_minus_index"] = range(0, len(hist))
    tminus_rows.append(hist)

tminus = pd.concat(tminus_rows, ignore_index=True, sort=False) if tminus_rows else pd.DataFrame()

keep_cols = [c for c in [
    "event_date", "event_ticker", "t_minus_index", "date", "ticker",
    "ret_1d", "ret_5d", "volume_ratio_20d", "close_vs_ma20", "volatility_20d",
    "buyer_dominance_ratio", "net_flow_ratio", "rank1_same_buyer_streak",
    "score_ara", "ara_signature_score_v1",
    "next_open_ret", "ara_ratio", "ara_opening_class"
] if c in tminus.columns]

if not tminus.empty and keep_cols:
    tminus[keep_cols].to_csv(OUTPUT_DIR / "strong_gap_tminus_event_study_rows_v2.csv", index=False)
    display(tminus[keep_cols])
else:
    print("No strong/near ARA events available for T-minus event study.")

if not tminus.empty:
    # IMPORTANT FIX:
    # t_minus_index is the group key, so it must NOT be included inside the median aggregation columns.
    # Otherwise reset_index() will try to create a second t_minus_index column and raise:
    # ValueError: cannot insert t_minus_index, already exists
    exclude_cols = {"event_date", "event_ticker", "t_minus_index", "date", "ticker", "ara_opening_class"}
    metric_cols = [
        c for c in keep_cols
        if c not in exclude_cols and pd.api.types.is_numeric_dtype(tminus[c])
    ]

    if metric_cols:
        med = (
            tminus.groupby("t_minus_index", as_index=False)[metric_cols]
            .median(numeric_only=True)
        )
        med.to_csv(OUTPUT_DIR / "strong_gap_tminus_event_study_median_v2.csv", index=False)
        display(med)
    else:
        print("No numeric metric columns available for T-minus median aggregation.")


,event_date,event_ticker,t_minus_index,date,ticker,ret_1d,ret_5d,volume_ratio_20d,close_vs_ma20,volatility_20d,buyer_dominance_ratio,net_flow_ratio,rank1_same_buyer_streak,score_ara,ara_signature_score_v1,next_open_ret,ara_ratio,ara_opening_class
0,2026-05-20,INTD,0,2026-05-20,INTD,0.243478,0.172131,4.657342,0.105955,0.064939,0.439049,0.000110,2,0.050528,0.909242,0.202797,0.811189,near_ara
1,2026-05-20,INTD,1,2026-05-19,INTD,-0.041667,-0.080000,7.447755,-0.106449,0.035292,0.399601,0.000310,1,0.150067,0.640338,0.000000,0.000000,normal_movement
2,2026-05-13,FITT,0,2026-05-13,FITT,-0.042056,0.004902,0.307691,-0.065634,0.107775,0.351245,0.000000,1,0.094538,0.520024,0.136585,0.546341,strong_gap_up
3,2026-05-22,DIVA,0,2026-05-22,DIVA,-0.122581,-0.133758,0.960188,-0.138695,0.057564,0.453453,-0.001262,2,0.876316,0.458509,0.176471,0.504202,strong_gap_up
4,2026-05-22,DIVA,1,2026-05-21,DIVA,0.115108,-0.077381,2.689108,-0.028822,0.051051,0.423461,0.004103,1,0.025142,0.833757,-0.019355,-0.055300,normal_movement
5,2026-05-22,DIVA,2,2026-05-20,DIVA,-0.027972,-0.201149,0.145809,-0.133146,0.042897,0.346600,0.000000,4,0.087209,0.365606,0.007194,0.020555,normal_movement
6,2026-05-22,DIVA,3,2026-05-19,DIVA,-0.071429,-0.100629,0.235779,-0.116466,0.048948,0.333478,-0.000058,3,0.142097,0.377874,-0.006993,-0.019980,normal_movement
7,2026-05-22,DIVA,4,2026-05-18,DIVA,-0.019108,0.000000,0.191389,-0.062405,0.092291,0.255401,-0.000962,2,0.068695,0.469466,0.006494,0.018553,normal_movement
8,2026-05-22,DIVA,5,2026-05-13,DIVA,-0.065476,-0.024845,0.169095,-0.040636,0.092146,0.316351,0.000000,1,0.126306,0.462236,0.000000,0.000000,normal_movement
9,2026-05-22,TALF,0,2026-05-22,TALF,0.248000,0.090909,2.068619,0.139518,0.109321,0.535002,0.000000,1,0.051335,0.899410,0.102564,0.410256,strong_gap_up


,t_minus_index,ret_1d,ret_5d,volume_ratio_20d,close_vs_ma20,volatility_20d,buyer_dominance_ratio,net_flow_ratio,rank1_same_buyer_streak,score_ara,ara_signature_score_v1,next_open_ret,ara_ratio
0,0,0.027523,0.004902,2.068619,-0.041096,0.107775,0.439049,0.000000,2.0,0.090767,0.755752,0.136585,0.504202
1,1,-0.041667,-0.080000,2.689108,-0.056277,0.051051,0.423461,0.000310,1.0,0.150067,0.640338,0.000000,0.000000
2,2,-0.027972,-0.201149,0.145809,-0.133146,0.042897,0.346600,0.000000,4.0,0.087209,0.365606,0.007194,0.020555
3,3,-0.071429,-0.100629,0.235779,-0.116466,0.048948,0.333478,-0.000058,3.0,0.142097,0.377874,-0.006993,-0.019980
4,4,-0.019108,0.000000,0.191389,-0.062405,0.092291,0.255401,-0.000962,2.0,0.068695,0.469466,0.006494,0.018553
5,5,-0.065476,-0.024845,0.169095,-0.040636,0.092146,0.316351,0.000000,1.0,0.126306,0.462236,0.000000,0.000000


## 16. Export final v2 dataset and research summary

In [36]:
# Save enriched dataset
work.to_csv(OUTPUT_DIR / "ara_opening_signature_dataset_v2.csv", index=False)
try:
    work.to_parquet(OUTPUT_DIR / "ara_opening_signature_dataset_v2.parquet", index=False)
except Exception as e:
    print("parquet export skipped:", repr(e))

summary = {
    "valid_labeled_rows": int(len(work)),
    "unique_dates": int(work["date"].nunique()),
    "unique_tickers": int(work["ticker"].nunique()),
    "class_distribution": work["ara_opening_class"].value_counts().to_dict(),
    "strong_gap_or_better_events": int(work["is_strong_gap_or_better"].sum()),
    "near_or_full_ara_events": int(work["is_near_or_full_ara"].sum()),
    "used_signature_weights": used_weights,
    "ara_model_score_col": ara_model_score_col,
    "output_dir": str(OUTPUT_DIR),
}

with open(OUTPUT_DIR / "research_summary_v2.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)

print(json.dumps(summary, indent=2, ensure_ascii=False, default=str))

{
  "valid_labeled_rows": 1165,
  "unique_dates": 6,
  "unique_tickers": 259,
  "class_distribution": {
    "normal_movement": 1160,
    "strong_gap_up": 4,
    "near_ara": 1
  },
  "strong_gap_or_better_events": 5,
  "near_or_full_ara_events": 1,
  "used_signature_weights": {
    "rank_volume_ratio_20d": 0.22,
    "rank_ret_1d": 0.18,
    "rank_ret_5d": 0.12,
    "rank_close_vs_ma20": 0.14,
    "rank_volatility_20d": 0.12,
    "rank_buyer_dominance_ratio": 0.12,
    "rank_net_flow_ratio": 0.07,
    "rank_rank1_same_buyer_streak": 0.03
  },
  "ara_model_score_col": "score_ara",
  "output_dir": "ara_opening_signature_research_v2_outputs"
}


## 17. Interpretation guide

Gunakan notebook ini untuk menjawab empat pertanyaan:

1. **Apakah `score_ara` top-k punya edge?**
   - Lihat `score_topk_edge_evaluation_v2.csv`.

2. **Event apa yang missed oleh `score_ara`?**
   - Lihat `event_capture_rank_by_score_v2.csv`.

3. **Apakah `ara_signature_score_v1` membantu menangkap missed event?**
   - Bandingkan daily rank event pada `score_ara` vs `ara_signature_score_v1` vs combined scores.

4. **Threshold feature apa yang terlihat menjanjikan?**
   - Lihat `probabilistic_threshold_discovery_v2.csv` dan `simple_feature_interaction_discovery_v2.csv`.

Batasan utama:

- Data sample pendek.
- Strong-gap/near-ARA event sangat sedikit.
- Ini bukan evidence final untuk production policy.
- Untuk dataset final, label next open sebaiknya diambil dari canonical OHLCV agar ticker yang hilang dari next all_scores tetap bisa diberi label.